
# Rare and Unseen Parent Values

An AnDE component estimates one distribution per combination of class and
super-parent value. When a combination is rare, or never appears in the
training data, that estimate is unreliable. Two parameters control what the
ensemble does about it.

**The Scenario:**
A continuous attribute $x$ separates the classes well. A categorical
attribute, the operating mode of a machine, takes the values A and B
almost always. Mode C appears only four times in training, always with
class 0. The example asks what the model predicts for an instance in
mode C as $x$ moves from the region of class 0 to the region of class 1.

**The Parameters:**

-   ``unseen_state``: what to use for a combination never seen in training,
    here (class 1, mode C). ``"laplace"`` uses the smoothed estimate of an
    empty cell, which for a continuous attribute is the pooled Gaussian of
    the whole dataset. ``"backoff"`` reuses the estimate of the class,
    $P(x \mid y)$, as Naive Bayes would.
-   ``min_parent_count``: the minimum number of training instances that a
    super-parent value needs for its component to take part in a prediction.
    Raising it removes the component of mode C altogether, the rule of the
    original AODE.


In [ ]:
# Author: The scikit-bayes Developers
# SPDX-License-Identifier: BSD-3-Clause

import matplotlib.pyplot as plt
import numpy as np

from skbn import AnDE

# --- 1. Generate Data ---
rng = np.random.RandomState(42)
n_samples = 600

y = rng.randint(0, 2, size=n_samples)
x = rng.normal(loc=np.where(y == 1, 1.5, -1.5), scale=1.0)
mode = rng.randint(0, 2, size=n_samples)  # Modes A (0) and B (1)

# Mode C (2): four instances, all of class 0
x_rare = rng.normal(loc=-1.5, scale=1.0, size=4)
X = np.column_stack(
    [np.concatenate([x, x_rare]), np.concatenate([mode, np.full(4, 2)])]
)
y = np.concatenate([y, np.zeros(4, dtype=int)])

# --- 2. Fit the Configurations ---
base = dict(n_dependence=1, categorical_features=[1])
configs = {
    "Laplace (Default)": (AnDE(unseen_state="laplace", **base), "indigo", "-"),
    "Back-Off to the Class": (AnDE(unseen_state="backoff", **base), "teal", "--"),
    "Minimum Parent Count of 30": (AnDE(min_parent_count=30, **base), "crimson", "-."),
}
for model, _, _ in configs.values():
    model.fit(X, y)

# Queries: x along its range, with the mode fixed
x_grid = np.linspace(-5, 5, 300)
queries = {
    "Mode A (Frequent)": np.column_stack([x_grid, np.zeros_like(x_grid)]),
    "Mode C (Four Training Instances, All of Class 0)": np.column_stack(
        [x_grid, np.full_like(x_grid, 2)]
    ),
}

# --- 3. Plotting ---
fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True)

for ax, (title, Q) in zip(axes, queries.items()):
    for name, (model, color, style) in configs.items():
        ax.plot(
            x_grid,
            model.predict_proba(Q)[:, 1],
            color=color,
            linestyle=style,
            linewidth=2.5,
            label=name,
        )
    ax.axhline(0.5, color="gray", linestyle=":", linewidth=1)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Continuous Attribute x")
    ax.grid(alpha=0.3)

# Training instances of mode C
axes[1].scatter(
    x_rare,
    np.full(4, 0.02),
    c="indigo",
    marker="o",
    s=70,
    edgecolors="w",
    zorder=5,
    label="Training Instances in Mode C",
)

axes[0].set_ylabel("P(Class 1)")
axes[1].legend(loc="upper left")

fig.suptitle("Predictions for a Frequent and for a Rare Parent Value", fontsize=16)
plt.tight_layout()
plt.subplots_adjust(top=0.87)
plt.show()